# Lecture 16: CAG — Cache-Augmented Generation

**Course:** NLP with LangChain | **Platform:** Hope to Skill  
**Duration:** ~20 minutes | **Level:** Strategic  

---

## What if you didn't need RAG at all?

In Lecture 10, you built a **Vanilla RAG** system. In Lecture 15, you built an **Agentic RAG** system.  
Both systems retrieve documents from a vector database, then generate answers.

> **But what if... you didn't need retrieval at all?**
>
> What if you just loaded ALL your documents into the LLM's context window once,  
> cached that context, and then answered questions instantly without any retrieval step?

**That's CAG: Cache-Augmented Generation.**

Based on research presented at WWW 2025, CAG is a fundamentally different approach:  
skip the vector database entirely, and use the LLM's massive context window instead.

### What You Will Learn

| # | Topic | Real-World Analogy |
|---|-------|--------------------|  
| 1 | The Core Insight of CAG | Reading the whole book vs searching an index |
| 2 | How KV Cache works | Your brain remembering without re-reading |
| 3 | CAG 3-Phase Process | Study once, answer questions forever |
| 4 | Benefits of CAG vs RAG | Zero retrieval latency, zero retrieval errors |
| 5 | Limitations of CAG | Context window ceiling, static knowledge |
| 6 | Ideal CAG use cases | HR policies, product docs, legal compliance |
| 7 | Choosing the right LLM | Claude 200K vs GPT 128K vs Gemini 1M |
| 8 | CAG cost analysis | When CAG becomes cheaper than RAG |

> **After this lecture:** You will understand when to use CAG instead of RAG,  
> and how to implement CAG for appropriate use cases.

### Key Resources

| Resource | URL |
|----------|-----|
| CAG Research Paper (WWW 2025) | https://arxiv.org/abs/2501.09284 |
| OpenAI API Documentation | https://platform.openai.com/docs/ |
| OpenAI Prompt Caching | https://platform.openai.com/docs/guides/prompt-caching |
| Anthropic Claude Documentation | https://docs.anthropic.com/ |
| Anthropic Prompt Caching | https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching |
| tiktoken (Token Counter) | https://github.com/openai/tiktoken |
| Gemini API Documentation | https://ai.google.dev/docs |

---

## 0. Environment Setup

Run this cell **once** to install the packages we need today.

**Requirements:**
- **OpenAI API key** — for the LLM (same as Lecture 10-15)
  - Get your key at: https://platform.openai.com/api-keys
- **For comparison:** We'll also compare CAG with RAG, so we'll use Qdrant and embeddings

In [1]:
# Install required packages (run once, then you can skip this cell)
# openai: OpenAI API for LLMs
# tiktoken: Token counting library for OpenAI models
# langchain: Core LangChain library
# langchain-openai: OpenAI integration
# langchain-text-splitters: Text splitting utilities
# langchain-community: Community integrations
# sentence-transformers: Local embeddings
# qdrant-client: Qdrant vector database client
# langchain-qdrant: Qdrant integration for LangChain
# langchain-huggingface: HuggingFace integration
#
# Documentation:
#   openai:      https://platform.openai.com/docs/
#   tiktoken:    https://github.com/openai/tiktoken
#   langchain:   https://python.langchain.com/docs/

%pip install openai tiktoken langchain langchain-openai langchain-text-splitters langchain-community sentence-transformers qdrant-client langchain-qdrant langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os

# ============================================================
# API KEY SETUP
# ============================================================

# OpenAI API key (required for this lecture)
# Get your key from: https://platform.openai.com/api-keys
os.environ["OPENAI_API_KEY"] = ""

# Verify key is set
openai_key = os.environ.get("OPENAI_API_KEY", "")

if openai_key and openai_key != "your-openai-api-key-here":
    print(f"[SUCCESS] OpenAI API key set: {openai_key[:8]}...")
else:
    print("[WARNING] Please set your OpenAI API key above!")
    print("          Get one at: https://platform.openai.com/api-keys")

[SUCCESS] OpenAI API key set: sk-proj-...


---

## 1. The Core Insight of CAG

### Modern LLMs Have HUGE Context Windows

| Model | Context Window | Equivalent Pages |
|-------|----------------|------------------|
| GPT-4o / GPT-4o-mini | 128,000 tokens | ~250-300 pages |
| Claude 3.5 Sonnet | 200,000 tokens | ~300-400 pages |
| GPT-4.1-mini | 1,000,000 tokens | ~1500-2000 pages |

**The Big Idea:**

```
Traditional RAG Pipeline:
  Question → Embed → Search Vector DB → Retrieve Chunks → Generate Answer
  (6 steps, multiple systems, latency: 500ms-2s)

CAG Approach:
  Question → Generate Answer (from cached full context)
  (1 step, single system, latency: 100-500ms)
```

### Why This Works

If your **entire knowledge base** fits in the context window (say, 50,000 tokens),  
why bother with:
- Chunking documents?
- Computing embeddings?
- Storing in a vector database?
- Retrieving relevant chunks?

**Just load everything into the LLM's context and cache it!**

### The Magic: KV Cache

When you send documents to an LLM, it creates an internal representation called a **KV (Key-Value) Cache**.  
This cache can be **reused** for subsequent queries without re-processing the documents.

> **Analogy:** Reading a book once and remembering it,  
> vs re-reading the book every time someone asks a question about it.

---

## 2. Token Counting: Will Your Docs Fit?

The first step in evaluating CAG is: **count your tokens**.

Let's use `tiktoken` to count how many tokens are in a document.

In [3]:
# ============================================================
# TOKEN COUNTING WITH TIKTOKEN
# ============================================================
# tiktoken is OpenAI's official token counting library.
# Use it to check if your documents fit in the context window.
#
# Documentation:
#   tiktoken: https://github.com/openai/tiktoken
#   OpenAI tokenizer: https://platform.openai.com/tokenizer
# ============================================================

import tiktoken

print("=" * 70)
print("TOKEN COUNTING DEMO")
print("=" * 70)

# Load our knowledge base document
with open("data/nlp_article.txt", "r", encoding="utf-8") as f:
    knowledge_text = f.read()

print(f"[INFO] Loaded document: data/nlp_article.txt")
print(f"[INFO] Character count: {len(knowledge_text):,}")

# Get the tokenizer for GPT-4o-mini
# Different models use different tokenizers!
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

# Encode the text to tokens
tokens = encoding.encode(knowledge_text)
token_count = len(tokens)

print(f"[SUCCESS] Token count: {token_count:,} tokens")
print()

# Compare against model context windows
print("Model Context Window Comparison:")
print("-" * 70)

models = [
    ("GPT-4o-mini", 128_000),
    ("GPT-4o / GPT-4 Turbo", 128_000),
    ("Claude 3.5 Sonnet", 200_000),
    ("Gemini 1.5 Pro", 1_000_000),
]

for model_name, context_window in models:
    utilization = (token_count / context_window) * 100
    fits = "YES" if token_count < context_window else "NO"
    print(f"{model_name:25} | {context_window:>10,} tokens | {utilization:>5.1f}% used | Fits: {fits}")

print("=" * 70)

# Rule of thumb: if your knowledge base uses < 50% of context window, CAG is viable
if token_count < 64_000:  # Half of 128K
    print("[SUCCESS] Your knowledge base is a GREAT candidate for CAG!")
    print("[INFO] It uses less than 50% of the context window.")
elif token_count < 128_000:
    print("[INFO] Your knowledge base MIGHT work with CAG.")
    print("[INFO] Consider leaving room for the question and answer.")
else:
    print("[WARNING] Your knowledge base is too large for CAG with most models.")
    print("[INFO] Consider RAG or hybrid approach instead.")

TOKEN COUNTING DEMO
[INFO] Loaded document: data/nlp_article.txt
[INFO] Character count: 8,039
[SUCCESS] Token count: 1,473 tokens

Model Context Window Comparison:
----------------------------------------------------------------------
GPT-4o-mini               |    128,000 tokens |   1.2% used | Fits: YES
GPT-4o / GPT-4 Turbo      |    128,000 tokens |   1.2% used | Fits: YES
Claude 3.5 Sonnet         |    200,000 tokens |   0.7% used | Fits: YES
Gemini 1.5 Pro            |  1,000,000 tokens |   0.1% used | Fits: YES
[SUCCESS] Your knowledge base is a GREAT candidate for CAG!
[INFO] It uses less than 50% of the context window.


---

## 3. How KV Cache Works

### What is KV Cache?

**KV Cache** = Key-Value cache inside the LLM's internal memory.

When the LLM processes your documents:
1. It creates an internal representation (keys and values for each token)
2. This representation is stored in GPU memory
3. For the next query, the LLM **reuses** this representation
4. Result: The LLM doesn't re-read the documents — it just "remembers" them

### The Flow

```
Phase 1: Initial Load (one-time cost)
  Documents → LLM → KV Cache Generated → Cached in Memory
  Time: ~2-5 seconds (depends on document size)

Phase 2: Query (repeated, fast)
  Question → LLM (uses cached KV) → Answer
  Time: ~100-500ms (no retrieval delay!)
```

### Latency Comparison

| Approach | Latency | Why |
|----------|---------|-----|
| **RAG** | 500ms - 2s | Embed query + vector search + retrieve chunks + LLM generation |
| **CAG** | 100ms - 500ms | LLM generation only (context already cached) |

**CAG can be 2-10x faster than RAG for subsequent queries.**

### Prompt Caching (OpenAI & Anthropic)

Both OpenAI and Anthropic support **Prompt Caching**:
- **OpenAI:** Automatically caches repeated prompts (no extra code needed)
- **Anthropic:** Explicit caching with `cache_control` parameter

**Documentation:**
- OpenAI Prompt Caching: https://platform.openai.com/docs/guides/prompt-caching
- Anthropic Prompt Caching: https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching

---

## 4. CAG Implementation: The 3-Phase Process

### Phase 1: Load Documents
Load all your documents into memory as a single text string.

### Phase 2: Create the CAG Query Function
Build a function that sends the **full context + question** to the LLM.

### Phase 3: Query
Ask questions! The LLM answers from the full context.

Let's implement it:

In [4]:
# ============================================================
# PHASE 1: LOAD ALL DOCUMENTS INTO MEMORY
# ============================================================
# No chunking, no embeddings, just raw text.
#
# Documentation:
#   Python file I/O: https://docs.python.org/3/tutorial/inputoutput.html
# ============================================================

print("=" * 70)
print("PHASE 1: LOAD DOCUMENTS")
print("=" * 70)

# Load the document
with open("data/nlp_article.txt", "r", encoding="utf-8") as f:
    knowledge_base = f.read()

print(f"[SUCCESS] Loaded document: {len(knowledge_base):,} characters")

# Count tokens
import tiktoken
encoding = tiktoken.encoding_for_model("gpt-4o-mini")
token_count = len(encoding.encode(knowledge_base))
print(f"[SUCCESS] Token count: {token_count:,} tokens")
print(f"[INFO] Fits in GPT-4o-mini context: {token_count < 128_000}")
print()
print("Knowledge base loaded and ready!")

PHASE 1: LOAD DOCUMENTS
[SUCCESS] Loaded document: 8,039 characters
[SUCCESS] Token count: 1,473 tokens
[INFO] Fits in GPT-4o-mini context: True

Knowledge base loaded and ready!


In [6]:
# ============================================================
# PHASE 2 & 3: CAG QUERY FUNCTION
# ============================================================
# This function sends the FULL knowledge base + question to the LLM.
# No retrieval step!
#
# Documentation:
#   OpenAI Chat Completions: https://platform.openai.com/docs/guides/chat-completions
#   LangChain ChatOpenAI: https://python.langchain.com/docs/integrations/chat/openai
# ============================================================

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import time

print("=" * 70)
print("PHASE 2 & 3: CAG QUERY FUNCTION")
print("=" * 70)

# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("[SUCCESS] LLM initialized: gpt-4o-mini")

# Create the CAG prompt template
# The entire knowledge base goes into the {context} variable
cag_prompt = ChatPromptTemplate.from_template(
    """You are a helpful teaching assistant for an NLP course.
Answer the question based ONLY on the following knowledge base.
If the knowledge base does not contain the answer, say "I don't have that information."

Knowledge Base:
{context}

Question: {question}

Answer:"""
)

# Create the CAG chain
cag_chain = cag_prompt | llm | StrOutputParser()

print("[SUCCESS] CAG chain created")
print()


def cag_query(question: str, knowledge: str, verbose: bool = True) -> tuple[str, float]:
    """Query the CAG system.

    Args:
        question: The question to ask
        knowledge: The full knowledge base text
        verbose: Whether to print progress

    Returns:
        (answer, latency_seconds)
    """
    if verbose:
        print(f"[QUERY] {question}")

    start_time = time.time()

    # Send the ENTIRE knowledge base + question to the LLM
    answer = cag_chain.invoke({
        "context": knowledge,
        "question": question,
    })

    latency = time.time() - start_time

    if verbose:
        print(f"[SUCCESS] Answer generated in {latency:.2f}s")
        print(f"[ANSWER] {answer}")
        print()

    return answer, latency


print("[SUCCESS] cag_query() function ready")
print()

# Test it!
print("=" * 70)
print("TEST: CAG QUERY")
print("=" * 70)

test_question = "What is Natural Language Processing?"
answer, latency = cag_query(test_question, knowledge_base)

print("=" * 70)
print("CAG system is ready to use!")
print("[INFO] No vector database needed!")
print("[INFO] No embeddings needed!")
print("[INFO] No chunking needed!")
print("[INFO] Just send the full context and query!")

PHASE 2 & 3: CAG QUERY FUNCTION
[SUCCESS] LLM initialized: gpt-4o-mini
[SUCCESS] CAG chain created

[SUCCESS] cag_query() function ready

TEST: CAG QUERY
[QUERY] What is Natural Language Processing?
[SUCCESS] Answer generated in 3.49s
[ANSWER] Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objective of NLP is to read, decipher, understand, and make sense of human language in a manner that is both valuable and meaningful. NLP combines computational linguistics — rule-based modeling of human language — with statistical, machine learning, and deep learning models, enabling computers to process human language in the form of text or voice data and to understand its full meaning, complete with the speaker's or writer's intent and sentiment.

CAG system is ready to use!
[INFO] No vector database needed!
[INFO] No embeddings needed!
[INFO] No chunking 

In [7]:
# ============================================================
# MULTIPLE QUERIES: DEMONSTRATING SPEED
# ============================================================
# Ask multiple questions to see the caching benefit.
# After the first query, subsequent queries are faster because
# the context is cached by OpenAI's servers.
# ============================================================

print("=" * 70)
print("MULTIPLE QUERIES TEST")
print("=" * 70)
print("[INFO] The first query may be slower (context not cached yet).")
print("[INFO] Subsequent queries should be faster (context cached).")
print()

test_questions = [
    "What is tokenization in NLP?",
    "What is the transformer architecture?",
    "How does BERT differ from GPT?",
    "What is attention mechanism?",
]

latencies = []

for i, question in enumerate(test_questions, 1):
    print(f"Query {i}/{len(test_questions)}: {question}")
    print("-" * 70)
    answer, latency = cag_query(question, knowledge_base, verbose=False)
    latencies.append(latency)
    print(f"[SUCCESS] Answered in {latency:.2f}s")
    print(f"[ANSWER] {answer[:150]}..." if len(answer) > 150 else f"[ANSWER] {answer}")
    print()

print("=" * 70)
print("LATENCY SUMMARY")
print("=" * 70)
for i, latency in enumerate(latencies, 1):
    print(f"Query {i}: {latency:.2f}s")

avg_latency = sum(latencies) / len(latencies)
print(f"\n[INFO] Average latency: {avg_latency:.2f}s")
print(f"[INFO] First query: {latencies[0]:.2f}s (context processing + generation)")
if len(latencies) > 1:
    avg_subsequent = sum(latencies[1:]) / len(latencies[1:])
    print(f"[INFO] Avg subsequent: {avg_subsequent:.2f}s (cached context)")
    speedup = latencies[0] / avg_subsequent if avg_subsequent > 0 else 1
    print(f"[SUCCESS] Speedup from caching: {speedup:.1f}x")

MULTIPLE QUERIES TEST
[INFO] The first query may be slower (context not cached yet).
[INFO] Subsequent queries should be faster (context cached).

Query 1/4: What is tokenization in NLP?
----------------------------------------------------------------------
[SUCCESS] Answered in 1.25s
[ANSWER] I don't have that information.

Query 2/4: What is the transformer architecture?
----------------------------------------------------------------------
[SUCCESS] Answered in 2.47s
[ANSWER] The transformer architecture, introduced in the landmark paper "Attention Is All You Need" by Vaswani et al. in 2017, revolutionized NLP. Unlike previ...

Query 3/4: How does BERT differ from GPT?
----------------------------------------------------------------------
[SUCCESS] Answered in 0.73s
[ANSWER] I don't have that information.

Query 4/4: What is attention mechanism?
----------------------------------------------------------------------
[SUCCESS] Answered in 0.90s
[ANSWER] I don't have that information.


#### What just happened?

We implemented CAG in just a few lines of code:

1. **Loaded** the full document into memory (no chunking!)
2. **Counted** tokens to verify it fits in the context window
3. **Created** a query function that sends the ENTIRE document to the LLM
4. **Asked** questions and got answers directly from the full context

**No retrieval. No vector database. No embeddings.**

The LLM has the **entire knowledge base** in context, so it can:
- Answer questions instantly
- Make connections across the entire document
- Never "miss" relevant information (because it sees everything)

**Trade-off:** This only works if your knowledge base fits in the context window!

---

## 5. Benefits of CAG vs RAG

| Feature | RAG | CAG |
|---------|-----|-----|
| **Retrieval Latency** | 200-800ms (embed + search) | 0ms (no retrieval!) |
| **Total Latency** | 2s - 10s | 1s - 5s |
| **Retrieval Errors** | Possible (wrong chunks) | None (sees everything) |
| **Architecture** | Complex (6 components) | Simple (just LLM) |
| **Dependencies** | Vector DB + Embeddings | None |
| **Cross-document reasoning** | Limited (only retrieved chunks) | Excellent (sees full context) |
| **Setup complexity** | High (DB, embeddings, chunking) | Low (load docs, query) |
| **Maintenance** | Vector DB management | Regenerate cache on updates |

### Key Advantages

1. **Zero Retrieval Latency:** No vector search means faster responses
2. **Zero Retrieval Errors:** The LLM sees all the documents, so it can't "miss" relevant info
3. **Simpler Architecture:** Just an LLM — no vector database, no embeddings, no chunking
4. **Better Cross-Document Reasoning:** The LLM can connect information across the entire knowledge base
5. **Easier Debugging:** If the answer is wrong, it's the LLM's fault — not a retrieval problem

### When CAG Shines

CAG is **ideal** when:
- Your knowledge base is **small** (<100K tokens)
- Information is **static** or rarely changes
- You have **high query volume** (cache cost is amortized)
- **Latency is critical** (real-time applications)
- You need **cross-document reasoning** (connecting info from multiple sources)

---

## 6. Limitations of CAG

| Limitation | Impact | Example |
|------------|--------|---------|
| **Context Window Ceiling** | Max ~200K tokens (most models) | Can't fit 1000-page manual |
| **Static Knowledge** | Must regenerate cache on updates | News site (updates hourly) won't work |
| **Upfront Cost** | Processing all docs has initial cost | Large docs = expensive first query |
| **Memory Cost** | Large KV caches need GPU RAM | May hit rate limits |
| **Not Real-Time** | Can't handle constantly changing data | Live data feeds won't work |

### The Context Window Problem

Even with Gemini's 1M token context:
- 1M tokens ≈ 1500-2000 pages
- Most enterprise knowledge bases are **much larger**
- Example: Wikipedia = billions of tokens (won't fit!)


### When NOT to Use CAG

❌ Large knowledge bases   
❌ Frequently updated data (daily or more often)  
❌ Real-time data feeds  
❌ Need for source citations (RAG can point to specific chunks)  
❌ Budget constraints (CAG can be more expensive per query for small volumes)

---

## 7. Ideal CAG Use Cases

| Use Case | Why CAG Works | Typical Size |
|----------|---------------|-------------|
| **Company HR/Legal Policies** | Changes annually, high query volume | 10-50K tokens |
| **Product Documentation** | Versioned, stable between releases | 20-80K tokens |
| **Small Business FAQ** | Rarely changes, simple queries | 5-20K tokens |
| **Legal/Compliance** | Static regulations, cross-doc reasoning | 30-100K tokens |
| **Course Materials** | Stable per semester, many students | 40-100K tokens |
| **Restaurant/Hotel Info** | Menu, services, FAQ — rarely changes | 5-15K tokens |

### Real-World Example: Company HR Policy Bot

**Scenario:**
- Company has 50 pages of HR policies
- Policies change once per year
- 500 employees × 10 queries/month = 5000 queries/month

**Why CAG wins:**
- ✅ Fits in context (50 pages ≈ 25-30K tokens)
- ✅ Static (annual updates)
- ✅ High query volume (amortizes cache cost)
- ✅ Cross-policy reasoning ("Does maternity leave stack with vacation?")
- ✅ Fast responses (employees want instant answers)


### Another Example: Product Documentation Bot

**Scenario:**
- SaaS product with 100-page docs
- Docs updated with each version release (monthly)
- Support team + customers = 10,000 queries/month

**Why CAG works:**
- ✅ Fits in context (100 pages ≈ 50-60K tokens)
- ✅ Relatively static (monthly updates are manageable)
- ✅ Very high query volume
- ✅ Customers want instant, accurate answers
- ✅ Cross-doc reasoning ("How do I integrate feature X with feature Y?")

---

## 8. Choosing the Right LLM for CAG

Not all LLMs are created equal for CAG. You need:
1. **Large context window** (128K minimum)
2. **Good "recall"** across long context (doesn't "forget" middle sections)
3. **Prompt caching support** (for cost efficiency)

### Model Comparison

| Model | Context Window | Recall Quality | Prompt Caching | Best For |
|-------|----------------|----------------|----------------|----------|
| **Claude 3.5 Sonnet** | 200K | Excellent | Yes | CAG (best overall) |
| **GPT-4o-mini** | 128K | Very Good | Yes | CAG (good balance) |
| **GPT-4.1-mini** | 1M | Good | Yes | CAG (established) |
| **Gemini 2.5 Pro** | 1M | Good | Yes | Very large docs |


**Documentation:**
- OpenAI Models: https://platform.openai.com/docs/models
- Anthropic Models: https://docs.anthropic.com/en/docs/about-claude/models
- Gemini Models: https://ai.google.dev/gemini-api/docs/models/gemini

---

## 10. CAG vs RAG Latency Benchmark

Let's compare CAG and RAG side-by-side on the same question.

In [ ]:
# ============================================================
# LATENCY COMPARISON: CAG VS RAG
# ============================================================
# Build a simple RAG system and compare it with CAG.
#
# Documentation:
#   Qdrant: https://qdrant.tech/documentation/
#   HuggingFace: https://huggingface.co/sentence-transformers
# ============================================================

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
import time

print("=" * 70)
print("CAG VS RAG LATENCY BENCHMARK")
print("=" * 70)

# Qdrant credentials
QDRANT_URL = ""
QDRANT_API_KEY = ""

# Build RAG system (simplified from Lecture 10)
print("[PROGRESS] Building RAG system...")

# Load and chunk documents
loader = TextLoader("data/nlp_article.txt", encoding="utf-8")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)

# Create embeddings and vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="cag_vs_rag_benchmark",
    force_recreate=True,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Create RAG chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

rag_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_prompt = ChatPromptTemplate.from_template(
    """Answer based on the context below.

Context:
{context}

Question: {question}

Answer:"""
)

print("[SUCCESS] RAG system ready")
print()

# Test question
test_question = "What is NLP?"

# ============================================================
# RUN RAG
# ============================================================
print("Testing RAG approach...")
print("-" * 70)

start = time.time()

# Retrieve
retrieve_start = time.time()
docs = retriever.invoke(test_question)
retrieve_time = time.time() - retrieve_start

# Generate
generate_start = time.time()
context = "\n\n".join([doc.page_content for doc in docs])
rag_chain = rag_prompt | rag_llm | StrOutputParser()
rag_answer = rag_chain.invoke({"context": context, "question": test_question})
generate_time = time.time() - generate_start

rag_total_time = time.time() - start

print(f"[RAG] Retrieval time: {retrieve_time:.3f}s")
print(f"[RAG] Generation time: {generate_time:.3f}s")
print(f"[RAG] Total time: {rag_total_time:.3f}s")
print(f"[RAG] Answer: {rag_answer[:100]}...")
print()

# ============================================================
# RUN CAG
# ============================================================
print("Testing CAG approach...")
print("-" * 70)

cag_answer, cag_time = cag_query(test_question, knowledge_base, verbose=False)

print(f"[CAG] Retrieval time: 0.000s (no retrieval!)")
print(f"[CAG] Generation time: {cag_time:.3f}s")
print(f"[CAG] Total time: {cag_time:.3f}s")
print(f"[CAG] Answer: {cag_answer[:100]}...")
print()

# ============================================================
# COMPARISON
# ============================================================
print("=" * 70)
print("COMPARISON SUMMARY")
print("=" * 70)

speedup = rag_total_time / cag_time

print(f"RAG total time:     {rag_total_time:.3f}s")
print(f"  - Retrieval:      {retrieve_time:.3f}s ({retrieve_time/rag_total_time*100:.0f}%)")
print(f"  - Generation:     {generate_time:.3f}s ({generate_time/rag_total_time*100:.0f}%)")
print()
print(f"CAG total time:     {cag_time:.3f}s")
print(f"  - Retrieval:      0.000s (0%)")
print(f"  - Generation:     {cag_time:.3f}s (100%)")
print()
print(f"[SUCCESS] CAG is {speedup:.1f}x faster than RAG")
print(f"[INFO] Latency reduction: {(rag_total_time - cag_time)*1000:.0f}ms")
print()
print("Why CAG is faster:")
print("  1. No embedding computation for the query")
print("  2. No vector search")
print("  3. No retrieval step at all")
print("  4. LLM already has the full context cached")

CAG VS RAG LATENCY BENCHMARK
[PROGRESS] Building RAG system...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 463.92it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[SUCCESS] RAG system ready

Testing RAG approach...
----------------------------------------------------------------------
[RAG] Retrieval time: 0.322s
[RAG] Generation time: 3.565s
[RAG] Total time: 3.888s
[RAG] Answer: Natural Language Processing (NLP) is a branch of artificial intelligence that focuses on the interac...

Testing CAG approach...
----------------------------------------------------------------------
[CAG] Retrieval time: 0.000s (no retrieval!)
[CAG] Generation time: 2.557s
[CAG] Total time: 2.557s
[CAG] Answer: Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focu...

COMPARISON SUMMARY
RAG total time:     3.888s
  - Retrieval:      0.322s (8%)
  - Generation:     3.565s (92%)

CAG total time:     2.557s
  - Retrieval:      0.000s (0%)
  - Generation:     2.557s (100%)

[SUCCESS] CAG is 1.5x faster than RAG
[INFO] Latency reduction: 1331ms

Why CAG is faster:
  1. No embedding computation for the query
  2. No vector sear

---

## 11. Mini Challenges

### Challenge 1: Multi-Document CAG
Load **multiple files** into a single CAG knowledge base.  
Hint: Concatenate multiple text files with separators.

### Challenge 2: CAG with Structured Output
Modify the `cag_query()` function to return answers as **JSON**.  
Example output: `{"answer": "...", "confidence": "high", "sources": ["section 1", "section 2"]}`

### Challenge 3: Token Budget Manager
Write a function `check_token_budget(files, max_tokens)` that:  
- Takes a list of file paths and a max token limit
- Counts total tokens across all files
- Returns whether they fit, and recommendations if they don't

In [ ]:
# ============================================================
# Challenge 1: Multi-document CAG
# ============================================================
# Load multiple files into a single knowledge base

# Your code here:


In [ ]:
# ============================================================
# Challenge 2: CAG with Structured Output
# ============================================================
# Return answers as JSON with answer, confidence, sources

# Your code here:


In [ ]:
# ============================================================
# Challenge 3: Token Budget Manager
# ============================================================
# Check if files fit in a token budget

# Your code here:


---

## 12. Quick Reference: CAG Cheat Sheet

### Setup Pattern

```python
# 1. Load documents
with open("knowledge.txt", "r") as f:
    knowledge = f.read()

# 2. Count tokens
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o-mini")
tokens = enc.encode(knowledge)
print(f"Tokens: {len(tokens)}")

# 3. Create CAG query function
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

def cag_query(question, knowledge):
    prompt = f"Knowledge:\n{knowledge}\n\nQuestion: {question}\n\nAnswer:"
    return llm.invoke(prompt).content

# 4. Query!
answer = cag_query("What is...?", knowledge)
```

### Key Concepts

| Concept | Definition |
|---------|------------|
| **CAG** | Cache-Augmented Generation - load full docs into LLM context |
| **KV Cache** | Internal LLM memory that caches processed context |
| **Context Window** | Max tokens an LLM can process at once |
| **Prompt Caching** | Reusing cached context across queries |
| **Token** | Basic unit of text for LLMs (~0.75 words) |

### Decision Guide

| Use CAG if... | Use RAG if... |
|---------------|---------------|
| Knowledge < 100K tokens | Knowledge > 100K tokens |
| Rarely changes | Changes frequently |
| Latency critical | Latency flexible |
| High query volume | Low query volume |
| Simple architecture | Need source citations |

### Useful Links

| Resource | URL |
|----------|-----|
| CAG Paper | https://arxiv.org/abs/2501.09284 |
| OpenAI API | https://platform.openai.com/docs/ |
| OpenAI Prompt Caching | https://platform.openai.com/docs/guides/prompt-caching |
| Anthropic Prompt Caching | https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching |
| tiktoken | https://github.com/openai/tiktoken |
| Gemini API | https://ai.google.dev/docs |

---

## 13. Key Takeaways

1. **CAG = Cache-Augmented Generation** — put everything in context, cache it, skip retrieval
2. **Modern LLMs have huge context windows** — 128K-200K tokens = 250-400 pages
3. **KV Cache enables speed** — context is processed once, reused for all queries
4. **Benefits:** Zero retrieval latency, zero retrieval errors, simpler architecture
5. **Limitations:** Context window ceiling, static knowledge, upfront cost
6. **Ideal use cases:** HR policies, product docs, FAQs — small, stable, high query volume
7. **Choose the right LLM:** Claude 3.5 (200K) or GPT-4o (128K) for best results
8. **Cost:** CAG becomes cheaper than RAG at high query volumes
9. **Speed:** CAG is 2-10x faster than RAG (no retrieval step)
10. **Decision:** Size + update frequency + latency = CAG or RAG

### The CAG vs RAG Decision

```
If knowledge < 100K tokens AND rarely changes AND latency matters:
    → Use CAG
Else:
    → Use RAG
```

### Next Lecture

**Lecture 17: RAG vs CAG — Making the Right Choice**

A systematic decision framework:  
- Head-to-head comparison  
- The 3 key decision questions  
- Decision flowchart  
- Hybrid architecture (combining both!)  
- Real-world examples and practical exercises

---

*Hope to Skill — Building the future, one skill at a time.*

---

## Appendix: PEP 8 Style Rules Used in This Notebook

All code in this notebook follows Python's PEP 8 style guide.

### Naming Conventions

| Rule | Convention | Example from This Notebook |
|------|-----------|----------------------------|
| Variables & functions | `snake_case` | `knowledge_base`, `cag_query`, `token_count` |
| Constants | `UPPER_CASE` | `QDRANT_URL`, `QDRANT_API_KEY`, `COST_PER_1K_INPUT` |
| Classes | `PascalCase` | `ChatOpenAI`, `ChatPromptTemplate`, `StrOutputParser` |

### Best Practices Used

| Practice | Why | Example |
|----------|-----|---------||
| Docstrings | Every function documented | `def cag_query(): """Query the CAG system."""` |
| f-strings | Clean formatting | `f"Tokens: {token_count:,}"` |
| Type hints | Clear function signatures | `def cag_query(question: str) -> tuple[str, float]` |
| Constants | No magic numbers | `MAX_TOKENS = 128_000` |
| Descriptive names | Self-documenting code | `calculate_cag_cost` not `calc_cost` |
| `.get()` with default | Safe dictionary access | `os.environ.get("KEY", "")` |